# 04 · Business translation — when to stop, who they are, what to do

All money figures use **assumed** prices, margins, call costs and capacity from `config.yaml`; every output
is labelled with them. Simulations replay the held-out test leads: a lead called produces its recorded
outcome (no treatment data → no causal or uplift claims; these are projections).
Logic: `src/economics.py`, `src/personas.py`.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import FIG_DIR, ROOT as REPO, load_config
from src.clean import load_raw, TARGET
from src.model import split
from src import economics as E, personas as P, eda

pd.set_option("display.max_colwidth", 90)
cfg = load_config()
df = load_raw()
train, test = split(df, cfg)
scored = pd.read_csv(REPO / "models" / "scored_leads.csv")
st = scored[scored.split == "test"].set_index("Prospect ID").loc[test["Prospect ID"]]
y, p = st[TARGET].to_numpy(), st["probability"].to_numpy()
A = E.Assumptions.from_config(cfg)
CUR = A.currency
print(A.label())
print(f"Test leads: {len(y):,} · baseline conversion {y.mean():.1%}")

ASSUMPTIONS: price ₹15,000, margin 60%, cost/call ₹150, capacity 200 calls/day (5 counsellors × 40)
Test leads: 2,310 · baseline conversion 38.5%


## 4a. Expected value per call and the profit-optimal cutoff

In [2]:
print(f"Value of one enrolment = price × margin = {CUR}{A.value_per_enrolment:,.0f}  (assumed)")
print(f"A call pays for itself when P(convert) > cost / value = {A.break_even_probability:.1%}")

n_ev = E.optimal_calls(p, A)
n_cap = E.optimal_calls(p, A, capacity=A.batch_capacity)
ev, cap = E.simulate(y, p, n_ev, A), E.simulate(y, p, n_cap, A)
everyone, rnd_cap = E.simulate_all(y, A), E.simulate_random(y, n_cap, A)

res = pd.DataFrame({
    "EV > 0 (no capacity limit)": ev,
    f"Capacity-limited ({A.batch_capacity:,} calls = {A.days_to_work_batch} days × {A.calls_per_day}/day)": cap,
    "Call everyone": everyone,
    f"Random {n_cap:,} calls": rnd_cap,
}).T[["calls", "pct_leads_called", "enrolments", "pct_enrolments_captured", "profit"]]
display(res.style.format({"calls": "{:,.0f}", "pct_leads_called": "{:.0%}", "enrolments": "{:,.0f}",
                          "pct_enrolments_captured": "{:.0%}", "profit": CUR + "{:,.0f}"}))
res.to_csv(REPO / "reports" / "cutoff_scenarios.csv")
print(f"Days to call every lead at {A.calls_per_day}/day: {len(y) / A.calls_per_day:.1f}")

Value of one enrolment = price × margin = ₹9,000  (assumed)
A call pays for itself when P(convert) > cost / value = 1.7%


,calls,pct_leads_called,enrolments,pct_enrolments_captured,profit
EV > 0 (no capacity limit),"2,223",96%,885,99%,"₹7,631,550"
"Capacity-limited (1,000 calls = 5 days × 200/day)","1,000",43%,689,77%,"₹6,051,000"
Call everyone,"2,310",100%,890,100%,"₹7,663,500"
"Random 1,000 calls","1,000",43%,385,43%,"₹3,317,532"


Days to call every lead at 200/day: 11.6


In [3]:
curve = E.profit_curve(y, p, A)
gain_cap = cap["profit"] - rnd_cap["profit"]
extra_enrol = cap["enrolments"] - rnd_cap["enrolments"]
title = (f"With {A.batch_capacity:,} calls for {len(y):,} leads, score order captures "
         f"{cap['pct_enrolments_captured']:.0%} of enrolments vs {rnd_cap['pct_enrolments_captured']:.0%} "
         f"in random order: {extra_enrol:,.0f} more enrolments, {CUR}{gain_cap/1e5:.1f} lakh more profit")
eda.plot_profit_curve(curve, title,
                      markers={"capacity limit": n_cap, "EV > 0 cutoff": n_ev},
                      currency=CUR, note=A.label() + ". Test leads replayed with recorded outcomes.",
                      path=FIG_DIR / "04_profit_curve.png")
print(title)

With 1,000 calls for 2,310 leads, score order captures 77% of enrolments vs 43% in random order: 304 more enrolments, ₹27.3 lakh more profit


**Reading this:** at these assumed prices a call breaks even at a very low probability, so the unconstrained
EV rule says to call nearly everyone. The binding constraint is **counsellor time**, not call cost. The model
earns its keep by deciding *which* leads get the limited calls.

## Sensitivity: price × cost per call (ASSUMED values)

In [4]:
prices = cfg["economics"]["sensitivity"]["course_price"]
costs = cfg["economics"]["sensitivity"]["cost_per_call"]
sens = E.sensitivity(y, p, A, prices, costs)
sens_cap = E.sensitivity(y, p, A, prices, costs, capacity=A.batch_capacity)
sens.to_csv(REPO / "reports" / "sensitivity_unconstrained.csv", index=False)
sens_cap.to_csv(REPO / "reports" / "sensitivity_capacity.csv", index=False)

def fmt(t):
    o = pd.DataFrame({
        "Price (assumed)": t.course_price.map(lambda v: f"{CUR}{v:,.0f}"),
        "Cost/call (assumed)": t.cost_per_call.map(lambda v: f"{CUR}{v:,.0f}"),
        "Break-even P": t.break_even_p.map("{:.1%}".format),
        "% leads called": t.pct_leads_called.map("{:.0%}".format),
        "% enrolments captured": t.pct_enrolments_captured.map("{:.0%}".format),
        "Profit: model (lakh)": (t.profit_model / 1e5).map("{:,.1f}".format),
        "vs call everyone (lakh)": (t.gain_vs_everyone / 1e5).map("{:+,.1f}".format),
        "vs random, same calls (lakh)": (t.gain_vs_random / 1e5).map("{:+,.1f}".format),
    })
    return o

print("UNCONSTRAINED — call every lead with EV > 0")
display(fmt(sens))
print(f"CAPACITY-LIMITED — at most {A.batch_capacity:,} calls for this batch")
display(fmt(sens_cap))

def to_md(t):
    return ("| " + " | ".join(t.columns) + " |\n|" + "---|" * len(t.columns) + "\n" +
            "\n".join("| " + " | ".join(map(str, r)) + " |" for r in t.itertuples(index=False)))
(REPO / "reports" / "sensitivity.md").write_text(
    f"# Sensitivity (ASSUMED prices and costs; margin {A.gross_margin:.0%})\n\n"
    f"Test set: {len(y):,} leads. 'vs call everyone' = model profit minus calling all {len(y):,} leads. "
    f"'vs random' = model profit minus calling the same number of leads in random order.\n\n"
    f"## Unconstrained (call while EV > 0)\n\n{to_md(fmt(sens))}\n\n"
    f"## Capacity-limited ({A.batch_capacity:,} calls: {A.counsellors} counsellors × "
    f"{A.calls_per_counsellor_per_day} calls/day × {A.days_to_work_batch} days)\n\n{to_md(fmt(sens_cap))}\n",
    encoding="utf-8")

UNCONSTRAINED — call every lead with EV > 0


,Price (assumed),Cost/call (assumed),Break-even P,% leads called,% enrolments captured,Profit: model (lakh),vs call everyone (lakh),"vs random, same calls (lakh)"
0,"₹5,000",₹50,1.7%,96%,99%,25.4,-0.1,+0.9
1,"₹5,000",₹150,5.0%,87%,98%,23.2,-0.0,+2.9
2,"₹5,000",₹300,10.0%,73%,96%,20.4,+0.6,+5.9
3,"₹15,000",₹50,0.6%,98%,100%,78.8,-0.2,+1.5
4,"₹15,000",₹150,1.7%,96%,99%,76.3,-0.3,+2.6
5,"₹15,000",₹300,3.3%,92%,99%,72.8,-0.4,+5.3
6,"₹30,000",₹50,0.3%,100%,100%,159.0,+0.0,+0.0
7,"₹30,000",₹150,0.8%,97%,100%,156.1,-0.6,+3.6
8,"₹30,000",₹300,1.7%,96%,99%,152.6,-0.6,+5.1


CAPACITY-LIMITED — at most 1,000 calls for this batch


,Price (assumed),Cost/call (assumed),Break-even P,% leads called,% enrolments captured,Profit: model (lakh),vs call everyone (lakh),"vs random, same calls (lakh)"
0,"₹5,000",₹50,1.7%,43%,77%,20.2,-5.4,+9.1
1,"₹5,000",₹150,5.0%,43%,77%,19.2,-4.1,+9.1
2,"₹5,000",₹300,10.0%,43%,77%,17.7,-2.1,+9.1
3,"₹15,000",₹50,0.6%,43%,77%,61.5,-17.4,+27.3
4,"₹15,000",₹150,1.7%,43%,77%,60.5,-16.1,+27.3
5,"₹15,000",₹300,3.3%,43%,77%,59.0,-14.2,+27.3
6,"₹30,000",₹50,0.3%,43%,77%,123.5,-35.5,+54.7
7,"₹30,000",₹150,0.8%,43%,77%,122.5,-34.2,+54.7
8,"₹30,000",₹300,1.7%,43%,77%,121.0,-32.2,+54.7


1837

In [5]:
worst = sens.gain_vs_everyone.min(); best = sens.gain_vs_everyone.max()
print(f"Unconstrained: the EV rule calls {sens.pct_leads_called.min():.0%}–{sens.pct_leads_called.max():.0%} "
      f"of leads; vs calling everyone it changes profit by {CUR}{worst:,.0f} to {CUR}{best:+,.0f}.")
print(f"Capacity-limited: vs random order with the same calls, the model adds "
      f"{CUR}{sens_cap.gain_vs_random.min()/1e5:.1f}–{sens_cap.gain_vs_random.max()/1e5:.1f} lakh across the grid.")
low = st.assign(p=p).query("p < @A.break_even_probability")
if len(low):
    print(f"Leads below break-even: {len(low)}; predicted {low.p.mean():.1%} vs actual {low[TARGET].mean():.1%} "
          f"(calibration is slightly pessimistic at the very bottom).")

Unconstrained: the EV rule calls 73%–100% of leads; vs calling everyone it changes profit by ₹-63,900 to ₹+64,200.
Capacity-limited: vs random order with the same calls, the model adds ₹9.1–54.7 lakh across the grid.
Leads below break-even: 87; predicted 0.7% vs actual 5.7% (calibration is slightly pessimistic at the very bottom).


## 4b. Personas

In [6]:
pm = P.fit_personas(train, random_state=cfg["model"]["random_state"])
print("Silhouette by k:", {k: round(v, 3) for k, v in pm.silhouettes.items()})
print("Chosen k =", pm.kmeans.n_clusters)
pt = P.persona_table(pm, test)
display(pt)
pt.to_csv(REPO / "reports" / "personas.csv")
joblib.dump(pm, REPO / "models" / "personas.joblib")

top2 = pt.head(2)
title = (f"{' and '.join(top2.index)} are {top2.share_of_leads.sum():.0%} of leads but "
         f"{top2.share_of_conversions.sum():.0%} of enrolments (test leads)")
eda.plot_personas(pt, y.mean(), title, FIG_DIR / "04_personas.png")

Silhouette by k: {3: 0.497, 4: 0.488, 5: 0.52, 6: 0.469}
Chosen k = 5


,leads,conversions,conv_rate,share_of_leads,share_of_conversions,lift,avg_minutes_on_site,playbook
persona,,,,,,,,
Form-fill referrals,168,151,0.898810,0.072727,0.169663,2.332865,3.198851,Call first: these sit at the top of every day's queue
Engaged working professionals,123,107,0.869919,0.053247,0.120225,2.257879,15.294751,Call first: these sit at the top of every day's queue
Website browsers,1463,514,0.351333,0.633333,0.577528,0.911886,10.556478,"Call in score order (Hot, then Warm)"
Chat-only drop-ins,422,105,0.248815,0.182684,0.117978,0.645801,0.303413,Call Hot/Warm in score order; put Cold into chat/WhatsApp follow-up
Email opt-outs,134,13,0.097015,0.058009,0.014607,0.251803,6.349803,Deprioritise: call only if Hot/Warm; otherwise low-cost nurture (no email)


<Figure size 800x450 with 1 Axes>

### How the score splits each persona

In [7]:
persona_test = pm.assign(test).to_numpy()
mix = pd.crosstab(persona_test, st["band"].to_numpy(), normalize="index").reindex(columns=["Hot", "Warm", "Cold"])
mix.index.name = "persona"
display(mix.style.format("{:.0%}"))
conv_by = pd.crosstab(persona_test, st["band"].to_numpy(), values=y, aggfunc="mean").reindex(columns=["Hot", "Warm", "Cold"])
display(conv_by.style.format("{:.0%}", na_rep="–"))

# Persona for every lead -> app
scored["persona"] = pm.assign(df.set_index(df.index)).to_numpy()
scored.to_csv(REPO / "models" / "scored_leads.csv", index=False)

col_0,Hot,Warm,Cold
persona,,,
Chat-only drop-ins,3%,40%,58%
Email opt-outs,0%,9%,91%
Engaged working professionals,93%,7%,1%
Form-fill referrals,94%,6%,0%
Website browsers,13%,32%,54%


col_0,Hot,Warm,Cold
row_0,,,
Chat-only drop-ins,55%,46%,9%
Email opt-outs,–,33%,7%
Engaged working professionals,90%,50%,0%
Form-fill referrals,91%,80%,–
Website browsers,70%,55%,14%


## 4c. Evidence for recommendations (all computed above)

In [8]:
EV = {}
# R1 — queue in score order under capacity
EV["r1"] = dict(capacity=A.batch_capacity, leads=len(y), model_capture=cap["pct_enrolments_captured"],
                random_capture=rnd_cap["pct_enrolments_captured"], extra_enrol=extra_enrol,
                extra_profit=gain_cap, per_100=(extra_enrol / len(y)) * 100)

# R2 — channel mix (from Phase 1 evidence, recomputed here on all leads)
from src.eda import segment_table
src_t = segment_table(df, "Lead Origin")
lsrc = segment_table(df.replace({"Lead Source": {"google": "Google"}}), "Lead Source")
EV["r2"] = dict(laf_rate=src_t.loc["Lead Add Form", "rate"], laf_share=src_t.loc["Lead Add Form", "share"],
                olark_rate=lsrc.loc["Olark Chat", "rate"], olark_share=lsrc.loc["Olark Chat", "share"],
                ref_rate=lsrc.loc["Reference", "rate"])

# Rejected idea (tested, kept here as evidence): skip non-Hot leads from low-lift personas
# and give their calls to the next leads in score order.
order = E.rank(p)
low_p = ["Email opt-outs", "Chat-only drop-ins"]
is_low = np.isin(persona_test, low_p)
skip = is_low & (st["band"].to_numpy() != "Hot")
keep_order = [i for i in order if not skip[i]]
enrol_score = y[order][:n_cap].sum()
enrol_skip = y[keep_order][:n_cap].sum()
print(f"Rejected: skipping low-lift personas (non-Hot) with {n_cap:,} calls -> {enrol_skip:.0f} enrolments "
      f"vs {enrol_score:.0f} in plain score order ({enrol_skip - enrol_score:+.0f}). The score already ranks them.")

# R3 — capacity: what would one more counsellor's calls bring?
extra_calls = A.calls_per_counsellor_per_day * A.days_to_work_batch
block = order[n_cap:n_cap + extra_calls]
blk_rate = y[block].mean()
blk_enrol = y[block].sum()
EV["r3"] = dict(extra_calls=extra_calls, block_rate=blk_rate, block_enrol=float(blk_enrol),
                block_profit=float(blk_enrol * A.value_per_enrolment - extra_calls * A.cost_per_call),
                break_even=A.break_even_probability, rank_from=n_cap + 1, rank_to=n_cap + extra_calls,
                rejected_persona_skip_delta=float(enrol_skip - enrol_score))
for k, v in EV.items():
    print(k, {kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in v.items()})
json.dump(EV, open(REPO / "reports" / "recommendation_evidence.json", "w"), indent=2, default=float)

Rejected: skipping low-lift personas (non-Hot) with 1,000 calls -> 672 enrolments vs 689 in plain score order (-17). The score already ranks them.
r1 {'capacity': 1000, 'leads': 2310, 'model_capture': np.float64(0.774), 'random_capture': 0.433, 'extra_enrol': 303.719, 'extra_profit': 2733467.532, 'per_100': 13.148}
r2 {'laf_rate': np.float64(0.925), 'laf_share': np.float64(0.078), 'olark_rate': np.float64(0.255), 'olark_share': np.float64(0.19), 'ref_rate': np.float64(0.918)}
r3 {'extra_calls': 200, 'block_rate': np.float64(0.35), 'block_enrol': 70.0, 'block_profit': 600000.0, 'break_even': 0.017, 'rank_from': 1001, 'rank_to': 1200, 'rejected_persona_skip_delta': -17.0}


In [9]:
r1, r2, r3 = EV["r1"], EV["r2"], EV["r3"]
recs = [
    {"action": f"Work every day's call list in lead-score order and stop at capacity "
               f"({A.calls_per_day} calls/day assumed); never call in arrival or random order.",
     "owner": "Sales ops (queue) + counsellors",
     "evidence": f"With {r1['capacity']:,} calls for {r1['leads']:,} test leads, score order reaches "
                 f"{r1['model_capture']:.0%} of enrolments vs {r1['random_capture']:.0%} at random.",
     "effect": f"Would add ~{r1['extra_enrol']:,.0f} enrolments per {r1['leads']:,} leads "
               f"(~{r1['per_100']:.0f} per 100 leads) with the same headcount; "
               f"{CUR}{r1['extra_profit']/1e5:.1f} lakh at assumed prices."},
    {"action": "Shift acquisition budget toward form-fill channels (Lead Add Form: Reference, Welingak) "
               "and test fixes for Olark Chat (qualifying questions in chat) before scaling it.",
     "owner": "Marketing",
     "evidence": f"Lead Add Form leads convert at {r2['laf_rate']:.0%} but are {r2['laf_share']:.0%} of volume; "
                 f"Olark Chat is {r2['olark_share']:.0%} of volume at {r2['olark_rate']:.0%}.",
     "effect": "More high-intent leads per rupee of spend, if quality holds at higher volume "
               "(retrospective data: validate with a small budget-shift test)."},
    {"action": "Treat counsellor capacity, not call cost, as the lever: before cutting calls, price one more "
               "counsellor (or a longer calling window) against the leads just below today's cutoff.",
     "owner": "Sales ops / sales management",
     "evidence": f"A call breaks even at {r3['break_even']:.1%} conversion (assumed prices); the next "
                 f"{r3['extra_calls']} leads below the {A.batch_capacity:,}-call cutoff (ranks {r3['rank_from']:,}–"
                 f"{r3['rank_to']:,}) still convert at {r3['block_rate']:.0%}.",
     "effect": f"One extra counsellor ({r3['extra_calls']} calls per batch) would reach ~{r3['block_enrol']:.0f} more "
               f"enrolments, {CUR}{r3['block_profit']/1e5:.1f} lakh before that counsellor's salary (assumed prices)."},
]
rec_df = pd.DataFrame(recs)
display(rec_df)
rec_df.to_csv(REPO / "reports" / "recommendations.csv", index=False)

,action,owner,evidence,effect
0,Work every day's call list in lead-score order and stop at capacity (200 calls/day ass...,Sales ops (queue) + counsellors,"With 1,000 calls for 2,310 test leads, score order reaches 77% of enrolments vs 43% at...","Would add ~304 enrolments per 2,310 leads (~13 per 100 leads) with the same headcount;..."
1,"Shift acquisition budget toward form-fill channels (Lead Add Form: Reference, Welingak...",Marketing,Lead Add Form leads convert at 92% but are 8% of volume; Olark Chat is 19% of volume a...,"More high-intent leads per rupee of spend, if quality holds at higher volume (retrospe..."
2,"Treat counsellor capacity, not call cost, as the lever: before cutting calls, price on...",Sales ops / sales management,A call breaks even at 1.7% conversion (assumed prices); the next 200 leads below the 1...,"One extra counsellor (200 calls per batch) would reach ~70 more enrolments, ₹6.0 lakh ..."
